#### Importar las librerías

In [1]:
import pandas as pd
import sys
import time
import mysql.connector

sys.path.insert(0,'./chromedriver-win64/chromedriver-win64/chromedriver.exe')
# pip install webdriver-manager
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
import os
from selenium.webdriver.common.keys import Keys

from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# Definimos una espera máxima de 10 segundos



In [25]:
pip install streamlit pandas plotly mysql-connector-python nltk

Note: you may need to restart the kernel to use updated packages.


#### Crear función de espera por cada click

In [2]:
def click_function(using, element):
    return wait.until(EC.element_to_be_clickable((using, element)))

#### Establecer directorio, carpeta de chromedriver y abrir página web IMDB

In [3]:
print(f"PATH: {os.listdir("./chromedriver-win64/chromedriver-win64")}")
service = Service("./chromedriver-win64/chromedriver-win64/chromedriver.exe")
#service = Service("chromedriver.exe")
options = webdriver.ChromeOptions()
options.add_argument("--incognito")

driver = webdriver.Chrome(service=service, options=options)
wait = WebDriverWait(driver, 25)
url = "https://www.imdb.com/es-es/"          
driver.get(url)

click_function(By.XPATH, '//*[@id="__next"]/div[1]/div/div[2]/div/button[1]').click()


PATH: ['.ipynb_checkpoints', 'chromedriver.exe', 'LICENSE.chromedriver', 'THIRD_PARTY_NOTICES.chromedriver']


#### Función para recolectar películas

In [4]:
def recolectar_peliculas(genero, filter_year, id_pelicula):
    from selenium.webdriver.common.by import By

    click_function(By.XPATH, '//*[@id="nav-search-form"]/div[1]/div/span[1]').click()
    click_function(By.XPATH, '//*[@id="nav-search-form"]/div[1]/div/div/div/div/ul/a').click()

    click_function(By.XPATH, "//span[text()='Desplegar todo'] | //span[text()='Plegar todo']").click()

    click_function(By.CSS_SELECTOR, 'button[data-testid="test-chip-id-movie"]').click()

    a = f'[data-testid="{filter_year}"]'
    
    # click_function(By.CSS_SELECTOR, '[data-testid="releaseYearMonth-end"]').send_keys(filter_year[1])
    fl_1 = f'[data-testid="{filter_year[0]}"]'
    click_function(By.CSS_SELECTOR, fl_1).send_keys(filter_year[1])

    tipo = f"button[data-testid='test-chip-id-{genero}']"
    boton = click_function(By.CSS_SELECTOR, tipo)
    driver.execute_script("arguments[0].click();", boton)
    
    
    click_function(By.XPATH, '//*[@id="__next"]/main/div[2]/div[3]/section/section/div/section/section/div[2]/div/section/div[1]/button').click()

    path_pelicula = f'//*[@id="__next"]/main/div[2]/div[3]/section/section/div/section/section/div[2]/div/section/div[2]/div[2]/ul/li[{id_pelicula}]/div/div/div/div[1]/div[2]/ul/div/a'
    link =  click_function(By.XPATH, path_pelicula)

    # 1. Obtener la URL del elemento
    url_pelicula = link.get_attribute("href")
    
    # 2. Guardar la pestaña actual
    pestana_principal = driver.current_window_handle
    
    # 3. Usar JavaScript para abrir la pestaña (esto hereda el modo incógnito)
    driver.execute_script(f'window.open("{url_pelicula}", "_blank");')
    
    # 4. Cambiar el foco a la nueva pestaña (la última en abrirse)
    time.sleep(2) # Espera breve para que el navegador reaccione
    driver.switch_to.window(driver.window_handles[-1])
    
    # --- extraer datos aquí ---
    
    # 5. Cerrar y volver

    
    list = []
    
    titulo = click_function(By.XPATH, '//*[@id="__next"]/main/div/section[1]/section/div[3]/section/section/div[2]/div[1]/h1').text
    # año = click_function(By.XPATH, '//*[@id="__next"]/main/div/section[1]/section/div[3]/section/section/div[2]/div[1]/ul/li[1]/a').text
    # año = click_function(By.XPATH, '//*[@id="__next"]/main/div/section[1]/section/div[3]/section/section/div[2]/div[1]/ul/li[1]/a').text
    año = click_function(By.XPATH, "//li[@role='presentation']/a[contains(@href, 'releaseinfo')]").text
    duracion = click_function(By.XPATH, '//*[@id="__next"]/main/div/section[1]/section/div[3]/section/section/div[2]/div[1]/ul/li[3]').text
    nota = driver.find_element(By.CSS_SELECTOR, 'div[data-testid$="score"] span').text
    try: 
        calificacion =  click_function(By.XPATH, '//*[@id="__next"]/main/div/section[1]/div/section/div/div[1]/section[5]/div[2]/div[1]/span[1]/span').text     
        director = click_function(By.XPATH, '//*[@id="__next"]/main/div/section[1]/div/section/div/div[1]/section[4]/ul/li[1]/div/ul/li/a').text
    except  Exception as e:
        calificacion =  click_function(By.XPATH, '//*[@id="__next"]/main/div/section[1]/section/div[3]/section/section/div[3]/div[2]/div[2]/div[1]/div/div[1]/a/span/div/div[2]/div[1]/span[1]').text  
        director = click_function(By.XPATH, '//*[@id="__next"]/main/div/section[1]/section/div[3]/section/section/div[3]/div[2]/div[2]/div[2]/ul/li[1]/div/ul/li/a').text
    
    # sinopsis = click_function(By.XPATH, '//*[@id="__next"]/main/div/section[1]/section/div[3]/section/section/div[3]/div[2]/div[1]/section/p/span[3]/span/span').text  
    sinopsis = click_function(By.XPATH, '//*[@id="__next"]/main/div/section[1]/section/div[3]/section/section/div[3]/div[2]/div[1]/section/p').text
    # pais = click_function(By.XPATH, '//*[@id="__next"]/main/div/section[1]/div/section/div/div[1]/section[11]/div[2]/ul/li[2]/div/ul/li/a').text
    
    # año = driver.find_element(By.XPATH, '//*[@id="__next"]/main/div/section[1]/section/div[3]/section/section/div[2]/div[1]/ul/li[1]/a').text
    # duracion = driver.find_element(By.XPATH, '//*[@id="__next"]/main/div/section[1]/section/div[3]/section/section/div[2]/div[1]/ul/li[3]').text
    # calificacion =  driver.find_element(By.XPATH, '//*[@id="__next"]/main/div/section[1]/div/section/div/div[1]/section[5]/div[2]/div[1]/span[1]/span').text     
    # director = driver.find_element(By.XPATH, '//*[@id="__next"]/main/div/section[1]/div/section/div/div[1]/section[4]/ul/li[1]/div/ul/li/a').text
    
    # reparto = "?"
    # sinopsis = driver.find_element(By.XPATH, '//*[@id="__next"]/main/div/section[1]/section/div[3]/section/section/div[3]/div[2]/div[1]/section/p/span[3]/span/span').text
    
    # pais = driver.find_element(By.XPATH, '//*[@id="__next"]/main/div/section[1]/div/section/div/div[1]/section[11]/div[2]/ul/li[2]/div/ul/li/a').text
    # pais = driver.find_element(By.XPATH, '//*[@id="__next"]/main/div/section[1]/div/section/div/div[1]/section[11]/div[2]/ul/li[2]/div/ul/li/a').text
    actors_elements = driver.find_elements(By.XPATH, '//*[@id="__next"]/main/div/section[1]/section/div[3]/section/section/div[3]/div[2]/div[2]/div[2]/ul/li[3]/div/ul//li/a')
    actores = []
    time.sleep(6)
    for actor in actors_elements:
        actores.append(actor.text)
        
    # actors_elements = click_function(By.XPATH, '//*[@id="__next"]/main/div/section[1]/section/div[3]/section/section/div[3]/div[2]/div[2]/div[2]/ul/li[3]/div/ul//li/a')
    # actores = []
    # for actor in actors_elements:
    #     actores.append(actor.text)
        

    boton = click_function(By.XPATH, "//span[text()='Reseñas de usuarios']")
    driver.execute_script("arguments[0].click();", boton)

    
    from selenium.webdriver.support.ui import WebDriverWait
    from selenium.webdriver.support import expected_conditions as EC
    
    
    # Esperar a que el select exista en el DOM
    select_el = WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.ID, "sort-by-selector"))
    )
    
    # Cambiar el valor con JS
    driver.execute_script("""
        const sel = arguments[0];
        sel.value = "SUBMISSION_DATE"; 
        sel.dispatchEvent(new Event('change', { bubbles: true }));
    """, select_el)

    reseña=[]
    for i in range(1,6):
        element = f'//*[@id="__next"]/main/div/section/div/section/div/div[1]/section[1]/article[{i}]/div[1]/div[1]'
        reseña_c = click_function(By.XPATH, element).text
        calif = reseña_c.split("\n")

        # print(calif)
        # califf = -1
        # if len(calif)>3:
        #     califf = calif[3]
            
        # if calif[3].lower()=='spoiler':
        #     califf=calif[2]
        reseña.append(calif)
        
    print(f"Título: {titulo}")
    print(f"Año: {año}")
    print(f"Duración: {duracion}")
    print(f"Calificación: {calificacion}")
    print(f"Director: {director}")
    print(f"Sinopsis: {sinopsis}")
    # print(f"País: {pais}")
    print(f"Actores: {actores}")
    print(f"Reseñas: {reseña}")
    
    info_pelis = {
        "Titulo": titulo,
        "Género": genero,
        "Año": año,
        "Duración": duracion,
        "Calificación": calificacion,
        "Director": director,
        "Protagonistas": actores,
        "Sinopsis": sinopsis,
        "Reseñas": reseña
    }
    time.sleep(1)
    driver.close()
    driver.switch_to.window(pestana_principal)
    return info_pelis

#### Recolección de películas

In [5]:
lista_pel = []
gen = ["Action", "Comedy", "Drama", "Horror", "Animation"]

for g in gen:
    print("=======================================================================")
    print("=======================================================================")
    print("=======================================================================")
    print(g)
    try:
        for i in range(1,12):
            print(f'<--------{i}-------->')
            filter_year = ["releaseYearMonth-end", 2024]
            info_pelis = recolectar_peliculas(g, filter_year, i)
            lista_pel.append(info_pelis)
    except Exception as e:
        time.sleep(1)
        driver.close()
        driver.switch_to.window(pestana_principal)
    time.sleep(3)

Action
<--------1-------->
Título: Noche de bodas
Año: 2019
Duración: 1h 35min
Calificación: 6,9
Director: Matt Bettinelli-Olpin
Sinopsis: La noche de bodas de una novia da un giro siniestro cuando sus nuevos suegros la obligan a formar parte de un juego aterrador.
Actores: ['', '', '']
Reseñas: [['7', '/10', 'Silly Premise, Seriously Entertaining - Dark Comedy Gold with an Explosive Payoff', "Stumbled upon this gem after hearing a sequel's on the way, and I'm genuinely excited for it.", '', "The premise is admittedly silly, but that's precisely the point... this dark comedy embraces its absurdity while delivering genuinely fresh thrills and plenty of satisfying gore. The explosive finale alone is worth the watch.", '', "Samara Weaving commands the screen with terrific presence, anchoring the entire film. Strong support from Adam Brody and Mark O'Brien rounds out the cast nicely. The only misfire is Andie MacDowell, oddly wasted in a role that doesn't utilize her talents effectively.",

#### Exportar dataset

In [6]:
import json


with open("movie.json", "w", encoding="utf-8") as f:
    json.dump(lista_pel, f, ensure_ascii=False, indent=2)


#### Importar base de datos

In [8]:
with open("movie.json", "r", encoding="utf-8") as f:
    data = json.load(f)

In [9]:
data

[{'Titulo': 'Noche de bodas',
  'Género': 'Action',
  'Año': '2019',
  'Duración': '1h 35min',
  'Calificación': '6,9',
  'Director': 'Matt Bettinelli-Olpin',
  'Protagonistas': ['', '', ''],
  'Sinopsis': 'La noche de bodas de una novia da un giro siniestro cuando sus nuevos suegros la obligan a formar parte de un juego aterrador.',
  'Reseñas': [['6',
    '/10',
    'A decently entertaining dark comedy horror',
    "Ready or Not is a perfectly fine film with a fun concept, I just don't think it takes it far enough!",
    '',
    'I had a lot of fun with the set up and the characters, but felt like there were a few missed opportunities. At times the story also seemed to lag a little bit and surrendered to cliche on a few too many occasions.',
    '',
    'Samara Weaving was a perfect scream queen and an easy character to root for and get on board with.',
    '',
    'Stylistically I liked the film, with its balance of horror, gore, and dark comedy. Again, it just could have been pushe

In [10]:
data[0]["Protagonistas"] = [
    "Samara Weaving",
    "Adam Brody",
    "Mark O'Brien"
]

#### Conectar base de datos

In [12]:
con = mysql.connector.connect(user = "root", password = "password", host = "localhost")
cursor = con.cursor()

In [13]:
cursor.execute("DROP DATABASE IF EXISTS peliculas")

#### Crear base de datos peliculas

In [14]:
cursor.execute("CREATE DATABASE peliculas")

cursor.execute("SHOW DATABASES")
for db in cursor:
    print(db)

('information_schema',)
('mysql',)
('peliculas',)
('performance_schema',)
('sys',)
('tarea',)
('tarea_bd',)


In [16]:
conexion2 = mysql.connector.connect(
    host="localhost",
    user="root",
    passwd="password",
    database="peliculas"
)

cursor1 = conexion2.cursor()

#### Crear tablas de películas y reseñas

In [17]:
crear_tabla_peliculas = """CREATE TABLE IF NOT EXISTS PELICULA
( 
   IDTITULO             int not null auto_increment,
   TITULO               longtext,
   GENERO               varchar(30),
   ANO                  year,
   DURACION             int,
   CALIFICACION         float,
   DIRECTOR             varchar(100),
   PROTAGONISTAS        longtext not null,
   SINOPSIS             longtext not null,
   primary key (IDTITULO)
);"""


crear_tabla_reseñas = """CREATE TABLE IF NOT EXISTS REVIEW
( 
   IDREVIEW             int not null auto_increment,   
   IDTITULO             int not null,
   CALIFICACION         float,
   REVIEW               longtext NOT NULL,
   primary key (IDREVIEW),
  constraint FK_PELICULA foreign key (IDTITULO)
      references PELICULA (IDTITULO) 
);"""

In [18]:
cursor1.execute(crear_tabla_peliculas)
cursor1.execute(crear_tabla_reseñas)

In [19]:
def order_reviews(title, review):
    rating = None
    rv = review

    if len(review) > 0:
        try:
            rating = float(review[0])
            rv = review[3:]
        except (ValueError, TypeError):
            rv = review

    rv_limpio = [str(x).replace("\n", " ").strip() for x in rv if str(x).strip() != ""]
    text = " ".join(rv_limpio)
    text = " ".join(text.split())

    return {
        "title": title,
        "rating": rating,
        "Reseña": text
    }


#### Cambiar duración a minutos

In [20]:
import re

def duracion_a_minutos(duracion):
    horas = 0
    minutos = 0

    h = re.search(r'(\d+)h', duracion)
    m = re.search(r'(\d+)min', duracion)

    if h:
        horas = int(h.group(1))
    if m:
        minutos = int(m.group(1))

    return horas * 60 + minutos

def limpiar_calificacion(calif):
    if calif is None:
        return None
    return float(calif.replace(",", "."))

#### Insertar datos en tabla de películas y reseñas

In [21]:
insert_pelicula = """
INSERT INTO PELICULA
(TITULO, GENERO, ANO, DURACION, CALIFICACION, DIRECTOR, PROTAGONISTAS, SINOPSIS)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
"""

In [22]:
insert_review = """
INSERT INTO REVIEW
(IDTITULO, CALIFICACION, REVIEW)
VALUES (%s, %s, %s)
"""

In [23]:
for movie in data:
    titulo = movie["Titulo"]
    genero = movie["Género"]
    ano = int(movie["Año"])
    duracion = duracion_a_minutos(movie["Duración"])
    calificacion = limpiar_calificacion(movie["Calificación"])
    director = movie["Director"]
    protagonistas = ", ".join([p for p in movie["Protagonistas"] if p.strip()])
    sinopsis = movie["Sinopsis"]

    cursor1.execute(insert_pelicula, (
        titulo,
        genero,
        ano,
        duracion,
        calificacion,
        director,
        protagonistas,
        sinopsis
    ))

    idtitulo = cursor1.lastrowid

    for review in movie["Reseñas"]:
        rev = order_reviews(movie["Titulo"], review)

        cursor1.execute(insert_review, (
            idtitulo,
            rev["rating"],
            rev["Reseña"]
        ))

conexion2.commit()

In [24]:
cursor1.close()
conexion2.close()
cursor.close()
con.close()
driver.quit()